In [1]:
from __future__ import annotations

import os
from dataclasses import dataclass
from getpass import getpass
from typing import Any

import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
COLLECTION_NAME = "ImportDocument"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

documents = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="external_id",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="title",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="content",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="category",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="source",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="priority",
            data_type=wvc.config.DataType.INT,
        ),
        wvc.config.Property(
            name="is_validated",
            data_type=wvc.config.DataType.BOOL,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: ImportDocument


In [5]:
raw_documents = [
    {
        "external_id": "doc-001",
        "title": "Weaviate Cloud Authentication",
        "content": "Weaviate Cloud requires a cluster URL, Weaviate API key and provider headers such as the OpenAI API key.",
        "category": "cloud",
        "source": "notebook",
        "priority": 5,
    },
    {
        "external_id": "doc-002",
        "title": "Vector Search Basics",
        "content": "Vector search uses embeddings to find semantically similar objects.",
        "category": "vector_search",
        "source": "notebook",
        "priority": 4,
    },
    {
        "external_id": "doc-003",
        "title": "Hybrid Search",
        "content": "Hybrid search combines BM25 keyword search with vector search using alpha.",
        "category": "hybrid_search",
        "source": "notebook",
        "priority": 4,
    },
    {
        "external_id": "doc-004",
        "title": "Focused RAG",
        "content": "Focused RAG works best with clean chunks from relevant notebooks and Markdown notes.",
        "category": "rag",
        "source": "markdown",
        "priority": 5,
    },
    {
        "external_id": "doc-005",
        "title": "Named Vectors",
        "content": "Named vectors allow one object to have multiple vector spaces such as title_vector and description_vector.",
        "category": "named_vectors",
        "source": "notebook",
        "priority": 3,
    },
    {
        "external_id": "doc-006",
        "title": "Multi-Tenancy",
        "content": "Multi-tenancy isolates data for different customers inside one shared collection schema.",
        "category": "multi_tenancy",
        "source": "notebook",
        "priority": 5,
    },
    {
        "external_id": "doc-007",
        "title": "Cross-References",
        "content": "Cross-references connect objects between collections using reference properties.",
        "category": "cross_references",
        "source": "notebook",
        "priority": 3,
    },
    {
        "external_id": "doc-008",
        "title": "Idempotent Imports",
        "content": "Deterministic UUIDs and upsert logic help avoid duplicate objects during repeated imports.",
        "category": "data_ingestion",
        "source": "notebook",
        "priority": 4,
    },

    # Invalid records below.
    {
        "external_id": "",
        "title": "Missing External ID",
        "content": "This record should fail validation because external_id is empty.",
        "category": "invalid",
        "source": "test",
        "priority": 1,
    },
    {
        "external_id": "doc-010",
        "title": "",
        "content": "This record should fail validation because title is empty.",
        "category": "invalid",
        "source": "test",
        "priority": 1,
    },
    {
        "external_id": "doc-011",
        "title": "Missing Content",
        "content": "",
        "category": "invalid",
        "source": "test",
        "priority": 1,
    },
    {
        "external_id": "doc-012",
        "title": "Wrong Priority Type",
        "content": "This record should fail validation because priority is not an integer.",
        "category": "invalid",
        "source": "test",
        "priority": "high",
    },
    {
        "external_id": "doc-013",
        "title": "Priority Out Of Range",
        "content": "This record should fail validation because priority is outside the accepted range.",
        "category": "invalid",
        "source": "test",
        "priority": 999,
    },
]

In [6]:
@dataclass
class ValidationResult:
    valid_items: list[dict[str, Any]]
    invalid_items: list[dict[str, Any]]

In [7]:
def validate_document(item: dict[str, Any]) -> tuple[bool, str | None]:
    required_text_fields = [
        "external_id",
        "title",
        "content",
        "category",
        "source",
    ]

    for field in required_text_fields:
        value = item.get(field)

        if not isinstance(value, str):
            return False, f"{field} must be a string"

        if not value.strip():
            return False, f"{field} cannot be empty"

    priority = item.get("priority")

    if not isinstance(priority, int):
        return False, "priority must be an integer"

    if priority < 1 or priority > 5:
        return False, "priority must be between 1 and 5"

    return True, None

In [8]:
def validate_documents(items: list[dict[str, Any]]) -> ValidationResult:
    valid_items = []
    invalid_items = []

    for item in items:
        is_valid, error = validate_document(item)

        if is_valid:
            validated_item = {
                **item,
                "external_id": item["external_id"].strip(),
                "title": item["title"].strip(),
                "content": item["content"].strip(),
                "category": item["category"].strip(),
                "source": item["source"].strip(),
                "is_validated": True,
            }
            valid_items.append(validated_item)
            continue

        invalid_items.append(
            {
                "item": item,
                "error": error,
            }
        )

    return ValidationResult(
        valid_items=valid_items,
        invalid_items=invalid_items,
    )

In [9]:
validation_result = validate_documents(raw_documents)

print("Valid items:", len(validation_result.valid_items))
print("Invalid items:", len(validation_result.invalid_items))

print("\nInvalid records:")
for invalid in validation_result.invalid_items:
    print("Error:", invalid["error"])
    print("Item:", invalid["item"])
    print("-" * 80)

Valid items: 8
Invalid items: 5

Invalid records:
Error: external_id cannot be empty
Item: {'external_id': '', 'title': 'Missing External ID', 'content': 'This record should fail validation because external_id is empty.', 'category': 'invalid', 'source': 'test', 'priority': 1}
--------------------------------------------------------------------------------
Error: title cannot be empty
Item: {'external_id': 'doc-010', 'title': '', 'content': 'This record should fail validation because title is empty.', 'category': 'invalid', 'source': 'test', 'priority': 1}
--------------------------------------------------------------------------------
Error: content cannot be empty
Item: {'external_id': 'doc-011', 'title': 'Missing Content', 'content': '', 'category': 'invalid', 'source': 'test', 'priority': 1}
--------------------------------------------------------------------------------
Error: priority must be an integer
Item: {'external_id': 'doc-012', 'title': 'Wrong Priority Type', 'content': '

In [10]:
result = documents.data.insert_many(validation_result.valid_items)

print("Has errors:", bool(result.errors))

if result.errors:
    print(result.errors)
else:
    print("Batch import completed.")

Has errors: False
Batch import completed.


In [11]:
response = documents.aggregate.over_all(total_count=True)

print("Imported objects:", response.total_count)

Imported objects: 8


In [12]:
response = documents.query.fetch_objects(
    limit=20,
    return_properties=[
        "external_id",
        "title",
        "category",
        "source",
        "priority",
        "is_validated",
    ],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print(obj.properties)
    print("-" * 80)

UUID: 29abd724-bd34-4507-9daa-42d4ec58b00b
{'source': 'markdown', 'title': 'Focused RAG', 'is_validated': True, 'external_id': 'doc-004', 'priority': 5, 'category': 'rag'}
--------------------------------------------------------------------------------
UUID: 36e9c6f0-483b-478c-91cf-60be54497372
{'source': 'notebook', 'title': 'Cross-References', 'is_validated': True, 'external_id': 'doc-007', 'priority': 3, 'category': 'cross_references'}
--------------------------------------------------------------------------------
UUID: 3971a55a-fefb-4fb8-b6b1-a0ba95f29e9b
{'source': 'notebook', 'title': 'Weaviate Cloud Authentication', 'is_validated': True, 'external_id': 'doc-001', 'priority': 5, 'category': 'cloud'}
--------------------------------------------------------------------------------
UUID: 52ee7966-4d8e-4b9b-ba71-679f9c19d0e7
{'source': 'notebook', 'title': 'Named Vectors', 'is_validated': True, 'external_id': 'doc-005', 'priority': 3, 'category': 'named_vectors'}
-------------------

In [13]:
def chunks(items: list[dict[str, Any]], size: int) -> list[list[dict[str, Any]]]:
    return [
        items[index : index + size]
        for index in range(0, len(items), size)
    ]

In [14]:
for number, chunk in enumerate(chunks(validation_result.valid_items, size=3), start=1):
    print("Chunk:", number, "Size:", len(chunk))

Chunk: 1 Size: 3
Chunk: 2 Size: 3
Chunk: 3 Size: 2


In [15]:
CHUNKED_COLLECTION_NAME = "ChunkedImportDocument"

if client.collections.exists(CHUNKED_COLLECTION_NAME):
    client.collections.delete(CHUNKED_COLLECTION_NAME)

chunked_documents = client.collections.create(
    name=CHUNKED_COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    properties=[
        wvc.config.Property(
            name="external_id",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="title",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="content",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="category",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="source",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="priority",
            data_type=wvc.config.DataType.INT,
        ),
        wvc.config.Property(
            name="is_validated",
            data_type=wvc.config.DataType.BOOL,
        ),
    ],
)

print("Created collection:", CHUNKED_COLLECTION_NAME)

Created collection: ChunkedImportDocument


In [16]:
import_report = {
    "chunks_total": 0,
    "objects_attempted": 0,
    "objects_imported": 0,
    "chunks_with_errors": 0,
    "errors": [],
}

for number, chunk in enumerate(chunks(validation_result.valid_items, size=3), start=1):
    print("Importing chunk:", number)

    import_report["chunks_total"] += 1
    import_report["objects_attempted"] += len(chunk)

    result = chunked_documents.data.insert_many(chunk)

    if result.errors:
        import_report["chunks_with_errors"] += 1
        import_report["errors"].append(
            {
                "chunk": number,
                "errors": result.errors,
            }
        )
        print("Chunk has errors.")
        continue

    import_report["objects_imported"] += len(chunk)
    print("Chunk imported:", len(chunk))

print(import_report)

Importing chunk: 1
Chunk imported: 3
Importing chunk: 2
Chunk imported: 3
Importing chunk: 3
Chunk imported: 2
{'chunks_total': 3, 'objects_attempted': 8, 'objects_imported': 8, 'chunks_with_errors': 0, 'errors': []}


In [17]:
def batch_import_with_report(
    collection,
    items: list[dict[str, Any]],
    *,
    chunk_size: int = 3,
) -> dict[str, Any]:
    report = {
        "chunks_total": 0,
        "objects_attempted": 0,
        "objects_imported": 0,
        "chunks_with_errors": 0,
        "errors": [],
    }

    for number, chunk in enumerate(chunks(items, size=chunk_size), start=1):
        report["chunks_total"] += 1
        report["objects_attempted"] += len(chunk)

        result = collection.data.insert_many(chunk)

        if result.errors:
            report["chunks_with_errors"] += 1
            report["errors"].append(
                {
                    "chunk": number,
                    "errors": result.errors,
                }
            )
            continue

        report["objects_imported"] += len(chunk)

    return report

In [18]:
REPORT_COLLECTION_NAME = "ReportedImportDocument"

if client.collections.exists(REPORT_COLLECTION_NAME):
    client.collections.delete(REPORT_COLLECTION_NAME)

reported_documents = client.collections.create(
    name=REPORT_COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(name="external_id", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="title", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="content", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="category", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="source", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="priority", data_type=wvc.config.DataType.INT),
        wvc.config.Property(name="is_validated", data_type=wvc.config.DataType.BOOL),
    ],
)

print("Created collection:", REPORT_COLLECTION_NAME)

Created collection: ReportedImportDocument


In [19]:
report = batch_import_with_report(
    reported_documents,
    validation_result.valid_items,
    chunk_size=4,
)

report

{'chunks_total': 2,
 'objects_attempted': 8,
 'objects_imported': 8,
 'chunks_with_errors': 0,
 'errors': []}

In [20]:
response = reported_documents.query.near_text(
    query="how to build RAG with clean chunks and notes",
    limit=5,
    return_properties=[
        "external_id",
        "title",
        "content",
        "category",
        "source",
        "priority",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("External ID:", obj.properties["external_id"])
    print("Title:", obj.properties["title"])
    print("Category:", obj.properties["category"])
    print("Priority:", obj.properties["priority"])
    print("Content:", obj.properties["content"])
    print("-" * 80)

Distance: 0.3941977024078369
External ID: doc-004
Title: Focused RAG
Category: rag
Priority: 5
Content: Focused RAG works best with clean chunks from relevant notebooks and Markdown notes.
--------------------------------------------------------------------------------
Distance: 0.7785818576812744
External ID: doc-008
Title: Idempotent Imports
Category: data_ingestion
Priority: 4
Content: Deterministic UUIDs and upsert logic help avoid duplicate objects during repeated imports.
--------------------------------------------------------------------------------
Distance: 0.7934235334396362
External ID: doc-006
Title: Multi-Tenancy
Category: multi_tenancy
Priority: 5
Content: Multi-tenancy isolates data for different customers inside one shared collection schema.
--------------------------------------------------------------------------------
Distance: 0.8010662794113159
External ID: doc-007
Title: Cross-References
Category: cross_references
Priority: 3
Content: Cross-references connect obj

In [21]:
response = reported_documents.query.fetch_objects(
    filters=Filter.by_property("priority").greater_or_equal(4),
    limit=20,
    return_properties=[
        "external_id",
        "title",
        "category",
        "priority",
    ],
)

for obj in response.objects:
    print(obj.properties)

{'title': 'Vector Search Basics', 'external_id': 'doc-002', 'priority': 4, 'category': 'vector_search'}
{'title': 'Weaviate Cloud Authentication', 'external_id': 'doc-001', 'category': 'cloud', 'priority': 5}
{'title': 'Hybrid Search', 'external_id': 'doc-003', 'priority': 4, 'category': 'hybrid_search'}
{'title': 'Focused RAG', 'external_id': 'doc-004', 'priority': 5, 'category': 'rag'}
{'title': 'Multi-Tenancy', 'external_id': 'doc-006', 'category': 'multi_tenancy', 'priority': 5}
{'title': 'Idempotent Imports', 'external_id': 'doc-008', 'category': 'data_ingestion', 'priority': 4}


In [22]:
response = reported_documents.generate.near_text(
    query="important Weaviate concepts for building applications",
    grouped_task=(
        "Summarize the retrieved imported documents. "
        "Focus on the most important Weaviate concepts. "
        "Mention the document titles and categories."
    ),
    limit=5,
    return_properties=[
        "external_id",
        "title",
        "content",
        "category",
        "priority",
    ],
)

print(response.generated)

print("\nSources:")
for obj in response.objects:
    print(
        "-",
        obj.properties["title"],
        "|",
        obj.properties["category"],
        "| priority:",
        obj.properties["priority"],
    )

Here’s a summarized overview of the most important Weaviate concepts based on the retrieved imported documents:

1. **Weaviate Cloud Authentication** (Category: Cloud)
   - Weaviate Cloud requires specific credentials for access, including a cluster URL and an API key for both Weaviate and the provider (e.g., OpenAI).

2. **Multi-Tenancy** (Category: Multi-Tenancy)
   - This feature ensures data isolation for different customers within a shared collection schema, allowing for secure and independent data handling.

3. **Named Vectors** (Category: Named Vectors)
   - Named vectors enhance the flexibility of Weaviate's data modeling by allowing a single object to host multiple vector spaces (e.g., `title_vector` and `description_vector`).

4. **Idempotent Imports** (Category: Data Ingestion)
   - The use of deterministic UUIDs and upsert logic in data ingestion prevents the duplication of objects during repeated import processes.

5. **Vector Search Basics** (Category: Vector Search)
   -

In [23]:
fixed_documents = [
    {
        "external_id": "doc-009",
        "title": "Fixed External ID",
        "content": "This record was fixed by adding a valid external_id.",
        "category": "fixed",
        "source": "retry",
        "priority": 1,
    },
    {
        "external_id": "doc-010",
        "title": "Fixed Missing Title",
        "content": "This record was fixed by adding a valid title.",
        "category": "fixed",
        "source": "retry",
        "priority": 1,
    },
    {
        "external_id": "doc-011",
        "title": "Fixed Missing Content",
        "content": "This record was fixed by adding valid content.",
        "category": "fixed",
        "source": "retry",
        "priority": 1,
    },
    {
        "external_id": "doc-012",
        "title": "Fixed Wrong Priority Type",
        "content": "This record was fixed by changing priority from text to integer.",
        "category": "fixed",
        "source": "retry",
        "priority": 2,
    },
    {
        "external_id": "doc-013",
        "title": "Fixed Priority Range",
        "content": "This record was fixed by setting priority inside the accepted range.",
        "category": "fixed",
        "source": "retry",
        "priority": 2,
    },
]

In [24]:
retry_validation_result = validate_documents(fixed_documents)

print("Retry valid items:", len(retry_validation_result.valid_items))
print("Retry invalid items:", len(retry_validation_result.invalid_items))

retry_report = batch_import_with_report(
    reported_documents,
    retry_validation_result.valid_items,
    chunk_size=2,
)

retry_report

Retry valid items: 5
Retry invalid items: 0


{'chunks_total': 3,
 'objects_attempted': 5,
 'objects_imported': 5,
 'chunks_with_errors': 0,
 'errors': []}

In [25]:
response = reported_documents.aggregate.over_all(total_count=True)

print("Final imported objects:", response.total_count)

Final imported objects: 13


In [26]:
response = reported_documents.query.fetch_objects(
    filters=Filter.by_property("source").equal("retry"),
    limit=20,
    return_properties=[
        "external_id",
        "title",
        "category",
        "source",
        "priority",
    ],
)

for obj in response.objects:
    print(obj.properties)

{'source': 'retry', 'title': 'Fixed Missing Title', 'external_id': 'doc-010', 'category': 'fixed', 'priority': 1}
{'source': 'retry', 'title': 'Fixed External ID', 'external_id': 'doc-009', 'priority': 1, 'category': 'fixed'}
{'source': 'retry', 'title': 'Fixed Wrong Priority Type', 'external_id': 'doc-012', 'priority': 2, 'category': 'fixed'}
{'source': 'retry', 'title': 'Fixed Missing Content', 'external_id': 'doc-011', 'category': 'fixed', 'priority': 1}
{'source': 'retry', 'title': 'Fixed Priority Range', 'external_id': 'doc-013', 'category': 'fixed', 'priority': 2}


In [27]:
client.close()